# CellAgent SCA Pipeline

统一入口：在 `cellagent` kernel 中一键执行预处理、QC report、通过 scGPT feature service 提取 encoder 特征、基于特征聚类、DE 分析。路径和参数从 `config/config.yaml` 读取。

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import time
import requests
import yaml

PROJECT_ROOT = Path('/root/wanghaoran/zxy/project/cellagent')
CONFIG = PROJECT_ROOT / 'config/config.yaml'
cfg = yaml.safe_load(CONFIG.read_text())
pipeline_cfg = cfg.get('pipeline', {})

INPUT_H5AD = Path(pipeline_cfg['input_path'])
OUTPUT_ROOT = Path(pipeline_cfg['output_root'])

PREPROCESSED_DIR = OUTPUT_ROOT / 'preprocessed'
QC_REPORT_DIR = OUTPUT_ROOT / 'qc_report'
FEATURE_DIR = OUTPUT_ROOT / 'features'
CLUSTER_DIR = OUTPUT_ROOT / 'clustering'
DE_DIR = OUTPUT_ROOT / 'de'
MULTIMODAL_PRIOR_DIR = OUTPUT_ROOT / 'multimodal_prior'
REASONING_DIR = OUTPUT_ROOT / 'reasoning'
FINAL_DIR = OUTPUT_ROOT / 'final'
MANIFEST = OUTPUT_ROOT / 'pipeline_manifest.json'

STEM = INPUT_H5AD.name[:-5] if INPUT_H5AD.name.endswith('.h5ad') else INPUT_H5AD.stem
PREPROCESSED_H5AD = PREPROCESSED_DIR / f'{STEM}_preprocessed.h5ad'
FEATURE_NPZ = FEATURE_DIR / f'{STEM}_preprocessed_cell_features.npz'
CLUSTERS_CSV = CLUSTER_DIR / f'{STEM}_preprocessed_cell_features_cell_clusters.csv'
CLUSTER_METRICS = CLUSTER_DIR / f'{STEM}_preprocessed_cell_features_clustering_metrics.json'

for path in [PREPROCESSED_DIR, QC_REPORT_DIR, FEATURE_DIR, CLUSTER_DIR, DE_DIR, MULTIMODAL_PRIOR_DIR, REASONING_DIR, FINAL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    'python': sys.executable,
    'config': str(CONFIG),
    'input': str(INPUT_H5AD),
    'output_root': str(OUTPUT_ROOT),
    'preprocessed': str(PREPROCESSED_H5AD),
    'feature_npz': str(FEATURE_NPZ),
    'clusters_csv': str(CLUSTERS_CSV),
    'feature_service': cfg.get('feature_service', {}),
}, ensure_ascii=False, indent=2))


In [ ]:
def run_cmd(cmd, skip_if_exists=None):
    if skip_if_exists is not None and Path(skip_if_exists).exists():
        print(f'[skip] exists: {skip_if_exists}')
        return
    full_cmd = [sys.executable] + [str(x) for x in cmd]
    print('[run]', ' '.join(full_cmd))
    subprocess.run(full_cmd, cwd=PROJECT_ROOT, check=True)

def require_exists(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    print(f'[ok] {path} ({path.stat().st_size} bytes)')

def require_feature_service():
    service_cfg = cfg.get('feature_service', {})
    if not service_cfg.get('enabled', False):
        raise RuntimeError('feature_service.enabled must be true for one-click cellagent notebook execution.')
    url = service_cfg.get('url', 'http://127.0.0.1:18080').rstrip('/')
    resp = requests.get(f'{url}/health', timeout=10)
    resp.raise_for_status()
    print('[feature-service]', json.dumps(resp.json(), ensure_ascii=False))
    return url


In [ ]:
# 1. Preprocess + QC report. Runs in the current cellagent kernel environment.
if PREPROCESSED_H5AD.exists():
    print(f'[skip] exists: {PREPROCESSED_H5AD}')
else:
    run_cmd([
        'scripts/preprocess_external_data.py',
        '--input', INPUT_H5AD,
        '--output-dir', PREPROCESSED_DIR,
        '--qc-report-dir', QC_REPORT_DIR,
        '--config', CONFIG,
    ])
require_exists(PREPROCESSED_H5AD)

if not (QC_REPORT_DIR / 'preprocessing_summary.json').exists():
    run_cmd([
        'scripts/write_qc_report.py',
        '--input', PREPROCESSED_H5AD,
        '--output-dir', QC_REPORT_DIR,
    ])
require_exists(QC_REPORT_DIR / 'preprocessing_summary.json')
require_exists(QC_REPORT_DIR / 'qc_metrics_summary.csv')


In [ ]:
# 2. Cell encoder feature extraction through the scGPT feature service.
# The notebook stays in the cellagent environment; scGPT/torch dependencies live behind the service.
require_feature_service()
run_cmd([
    'scripts/request_feature_service.py',
    '--input', PREPROCESSED_H5AD,
    '--feature-dir', FEATURE_DIR,
    '--config', CONFIG,
], skip_if_exists=FEATURE_NPZ)
require_exists(FEATURE_NPZ)


In [ ]:
# 3. Leiden clustering on extracted cell encoder features.
run_cmd([
    'scripts/cluster_cell_features.py',
    '--features', FEATURE_NPZ,
    '--output-dir', CLUSTER_DIR,
    '--config', CONFIG,
], skip_if_exists=CLUSTERS_CSV)
require_exists(CLUSTERS_CSV)
require_exists(CLUSTER_METRICS)
print(json.dumps(json.loads(CLUSTER_METRICS.read_text()), ensure_ascii=False, indent=2)[:2000])


In [ ]:
# 4. DE analysis by cluster. Saves up to config.de_analysis.top_k significant genes per cluster.
de_summary = DE_DIR / 'de_summary.csv'
run_cmd([
    'scripts/run_de_analysis.py',
    '--preprocessed', PREPROCESSED_H5AD,
    '--clusters', CLUSTERS_CSV,
    '--output-dir', DE_DIR,
    '--config', CONFIG,
], skip_if_exists=de_summary)
require_exists(de_summary)
print(de_summary.read_text().splitlines()[:12])


In [ ]:
# 5. Write the stable artifact manifest for downstream multimodal prior / reasoning / judge notebooks.
run_cmd([
    'scripts/write_pipeline_manifest.py',
    '--config', CONFIG,
    '--output', MANIFEST,
    '--preprocessed-h5ad', PREPROCESSED_H5AD,
    '--qc-report-dir', QC_REPORT_DIR,
    '--feature-npz', FEATURE_NPZ,
    '--clusters-csv', CLUSTERS_CSV,
    '--clustering-metrics-json', CLUSTER_METRICS,
    '--de-summary-csv', de_summary,
    '--de-dir', DE_DIR,
    '--reasoning-dir', REASONING_DIR,
    '--final-dir', FINAL_DIR,
])
require_exists(MANIFEST)
print(json.dumps(json.loads(MANIFEST.read_text()), ensure_ascii=False, indent=2)[:3000])
